In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict, Literal
from langgraph.graph.message import add_messages

class MedicalState(TypedDict):
    message: Annotated[list, add_messages]
    next: Literal[
        "diagnostic_agent",
        "physician_review",
        "report_agent",
        "END"
    ]
    question_count: int
    interim_care: str
    diagnostic_summary: str
    physician_treatment: str
    final_report: str

In [ ]:
from state import MedicalState
from typing_extensions import Literal
from IPython.display import Markdown


def supervisor(state:MedicalState) -> dict:
    if state.get("final_report"):
        display(Markdown(state["final_report"]))
        return {"next" : "END"}
    if state["question_count"] < 5:
        return {"next" : "diagnostic_agent"}
    if state["question_count"] == 5:
        return {"next" : "report_agent"}



def route_after_supervisor(state: MedicalState) -> Literal["diagnostic_agent","report_agent","END"]:
    return state["next"]
    



In [ ]:
from state import MedicalState
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage


diagnostic_agent_llm = init_chat_model("openai:gpt-5-mini")

MAX_QUESTIONS = 5

DIAGNOSTIC_SYSTEM_PROMPT = (
    "You are a diagnostic intake agent for a medical triage assistant. "
    "Ask the patient exactly one short, clear, relevant question at a time "
    "to understand their symptoms, history, and condition, building on "
    "their previous answers. Reply with only the question itself — no "
    "preamble, no diagnosis, no advice."
)


def DiagnosticAgent(state: MedicalState) -> dict:
    """Asks the patient one diagnostic question per invocation.

    Meant to be called once per question (MAX_QUESTIONS times total): it
    reads the conversation so far from state["message"] (the latest human
    message being the patient's most recent answer) and appends the agent's
    next question as a new AI message, bumping question_count by one.

    Note: diagnostic_summary is NOT produced here. By the time this node
    would see question_count == MAX_QUESTIONS, the patient's answer to the
    final question hasn't been added to state yet, and nothing currently
    calls this node again afterward to pick it up — the supervisor routes
    away as soon as question_count == 5. Summarizing needs its own step
    once the graph wiring (and the supervisor's routing for counts 1-4,
    which it doesn't handle yet either) is decided.
    """
    question_count = state.get("question_count", 0)
    history = [SystemMessage(content=DIAGNOSTIC_SYSTEM_PROMPT), *state["message"]]

    if question_count == MAX_QUESTIONS - 1:
        history.append(SystemMessage(
            content="This is the final question — after it, no more will be asked."
        ))

    question = diagnostic_agent_llm.invoke(history)

    print(question.content)
    return {
        "message": [question],
        "question_count": question_count + 1,
    }


In [ ]:
from ddgs import DDGS
from langchain.tools import tool


@tool
def web_search(query: str, num_results:int=5) -> str:
    """
        Search the web usin DuckDuckGo
        Args:
        query : Search query string
            num_results : Number of results to return (Default=5)
        Returns:
        Formatted search results with titles, descptions and Urls
    """
    try:
        print(f'Search tool is called for {query}')
        ddgs_search = DDGS()
        results=ddgs_search.text(query=query, max_results=num_results, backend="google")
        if not results:
            return f"No results found for {query} "
        formatted_results = [f"Search for {query} : \n"]
        for i, result in enumerate(results,1):
            title = result.get("title","No Title")
            body = result.get("body","No description available")
            href = result.get("href","")
        formatted_results.append( f"{i}. **{title}**. {body}. {href}")
        return formatted_results
    except Exception as e:
        print(str(e))

In [ ]:
report_agent_llm = init_chat_model("openai:gpt-5-mini").bind_tools([web_search])

REPORT_AGENT_PROMPT= "You are a reporter agent, your job is to generate a report about the patient situation based on {messages} propose solutions and medicaments."

def report_agent(state:MedicalState) -> dict:
    """
        You are a reporter agent, your job is to genarte a report to the patient
    """
    messages = state["message"]
    entretien = ""
    print("Inside the report agent")
    for p in messages:
        entretien = entretien+" "+p.content
    report = report_agent_llm.invoke(REPORT_AGENT_PROMPT.format(messages=entretien))
    state["question_count"] = state["question_count"] + 10
    return {"final_report":report.content}
    

In [ ]:
from langchain_core.messages import HumanMessage


def PatientResponse(state: MedicalState) -> dict:
    """Prompts the patient for their answer to the last question asked
    and stores it in state["message"] as a new human message.
    """
    answer = input("Your answer: ")
    return {"message": [HumanMessage(content=answer)]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, Markdown, display

builder = StateGraph(MedicalState)
builder.add_node("supervisor",supervisor)
builder.add_node("report_agent",report_agent)
builder.add_node("diagnostic_agent",DiagnosticAgent)
builder.add_edge(START, "supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {
        "diagnostic_agent": "diagnostic_agent",
        "report_agent": "report_agent",
        "END": END,
    },
)
builder.add_node("patient_response", PatientResponse)
builder.add_edge("diagnostic_agent", "patient_response")
builder.add_edge("patient_response", "supervisor")
builder.add_edge("report_agent","supervisor")



In [ ]:
graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"message": [], "question_count": 0})